# Model 05 — Decision Tree (Stamping Press Scrap Risk Classification)

Scrap in a stamping press line is not random. It is the product of specific, identifiable conditions — a new operator running a high-speed setup on a difficult material batch without completing the pre-run checklist. These factors interact in non-linear ways: velocity alone does not cause scrap, but velocity combined with an inexperienced operator and an incomplete setup checklist almost always does. This interaction structure makes scrap risk a natural classification problem, and Decision Trees a natural model choice.

**Decision Trees** are the most operationally transparent ML model in manufacturing. They produce explicit if-then rules that a process engineer can read, validate against plant knowledge, and post on the line without a data science degree. They identify thresholds — the speed at which risk jumps, the experience level that stabilizes a run, the checklist completion that gates the whole operation. These are the same thresholds that Six Sigma DMAIC projects try to identify through designed experiments. The tree finds them from historical data.

The dataset has **2,317 stamping press production records** across three risk levels — Low, Medium, and High scrap risk — based on 8 process and operator variables. The target is multiclass, which makes this project a step up in complexity from the binary classifiers in Models 01–04. We evaluate with macro F1 and per-class metrics to ensure the model performs across all three risk levels, not just the majority class.

---
**Data source:** `scrap_risk_data.csv`  
**Output:** metrics, tree visualization, text rules, Gini feature importance, 2D decision map, and a production scenario simulator

## Section 1 — Setup

We fix a global seed for reproducibility and import all dependencies upfront. Decision Trees are deterministic given the same seed and training data — essential when operational rules extracted from the tree are used in process documentation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. Random state:", RANDOM_STATE)


## Section 2 — Load Data

The dataset contains **2,317 stamping press production records** from real manufacturing environment. Each row represents one production run, described by 8 process and operator variables: press speed (spm), raw material hardness (HRB), operator experience (years), ambient temperature (°C), supplier lot quality flag, shift, recent model change flag, and setup checklist completion. The target `scrap_risk` is a three-class variable — Low, Medium, or High — assigned based on a composite risk score that reflects real stamping process behavior.

> *Note: The dataset was collected from different resources that represents stamping press conditions — where scrap risk emerges from combinations of factors: incomplete setup checklists and critical supplier lots are the strongest triggers; high press speed with inexperienced operators compounds the risk; night and early morning shifts introduce additional variability. The resulting class distribution (Low: 19.9%, Medium: 50.4%, High: 29.7%) reflects a real manufacturing environment where most runs carry some risk but only a subset are truly critical.*

In [ ]:
try:
    df = pd.read_csv("scrap_risk_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/Process_Decisions_Optimization/main/scrap_risk_data.csv")
    # FileNotFoundError is intentionally specific — a bare except would silently swallow real data errors

NUM_COLS = ["press_speed_spm", "raw_material_hardness_hrb", "operator_experience_yrs",
            "ambient_temp_c", "critical_supplier_lot", "recent_model_change",
            "setup_checklist_complete"]
CAT_COLS = ["shift"]
TARGET   = "scrap_risk"
CLASSES  = ["Low", "Medium", "High"]  # natural ordering for display

df.head()


## Section 3 — Quick Sanity Checks

We validate the loaded data before any modeling. This is a multiclass problem — three risk levels — so we check the class distribution carefully. With 50% of records in Medium, macro-averaged F1 is the right performance metric: it treats each class equally regardless of frequency, avoiding the illusion of high accuracy through majority-class dominance.


In [ ]:
# Sanity checks. Real datasets usually try to hurt you 🙂
print("Shape:", df.shape)
print("\n--- Data types ---")
df.info()


In [ ]:
print("--- Missing values ---")
print(df.isna().sum())
print("\n--- Target class distribution ---")
print(df[TARGET].value_counts())
print("\n--- Class rates ---")
print(df[TARGET].value_counts(normalize=True).round(3))


In [ ]:
print("--- Shift distribution ---")
print(df["shift"].value_counts())
print("\n--- Binary flags ---")
print("critical_supplier_lot:   ", df["critical_supplier_lot"].value_counts().to_dict())
print("recent_model_change:     ", df["recent_model_change"].value_counts().to_dict())
print("setup_checklist_complete:", df["setup_checklist_complete"].value_counts().to_dict())
print("\n--- Numeric summary ---")
df[["press_speed_spm","raw_material_hardness_hrb",
   "operator_experience_yrs","ambient_temp_c"]].describe().round(2)


## Section 4 — Exploratory Data Analysis

EDA for a multiclass problem requires visualizing how each feature distributes across all three risk levels — not just binary comparisons. We are looking for features that create visible separation between Low, Medium, and High risk classes. Features with the most separation will dominate the tree's top nodes; features with overlapping distributions will appear lower or not at all.

The categorical breakdown of risk rate by shift and binary flags is particularly actionable: it tells quality engineers which structural conditions are associated with high-risk runs before they even look at the model.


In [ ]:
# Class distribution bar chart
risk_order = ["Low", "Medium", "High"]
risk_colors = {"Low": "#4C9BE8", "Medium": "#F0A500", "High": "#E8574C"}

fig, ax = plt.subplots(figsize=(6, 4))
counts = df[TARGET].value_counts().reindex(risk_order)
bars = ax.bar(risk_order, counts.values,
              color=[risk_colors[r] for r in risk_order], edgecolor="white")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 10,
            f"{v}\n({v/len(df):.1%})", ha="center", fontweight="bold", fontsize=10)
ax.set_title("Scrap Risk Class Distribution", fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Scrap Risk Level")
plt.tight_layout()
plt.show()


In [ ]:
# Numeric distributions by risk class — boxplots across three levels
cont_cols = ["press_speed_spm", "raw_material_hardness_hrb",
             "operator_experience_yrs", "ambient_temp_c"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(cont_cols):
    for cls, color in risk_colors.items():
        order_idx = risk_order.index(cls)
        subset = df[df[TARGET] == cls][col]
        axes[i].boxplot([subset.values], positions=[order_idx],
                        patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.7),
                        medianprops=dict(color="white", linewidth=2),
                        widths=0.5)
    axes[i].set_xticks(range(len(risk_order)))
    axes[i].set_xticklabels(risk_order)
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xlabel("Scrap Risk Level")

plt.suptitle("Numeric Feature Distribution by Risk Class",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Risk rate by categorical and binary conditions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Shift
shift_risk = df.groupby("shift")[TARGET].value_counts(normalize=True).unstack().reindex(columns=risk_order, fill_value=0)
shift_risk.plot(kind="bar", ax=axes[0], color=[risk_colors[r] for r in risk_order],
                edgecolor="white", rot=20)
axes[0].set_title("Risk Rate by Shift")
axes[0].set_ylabel("Proportion")
axes[0].legend(title="Risk", fontsize=8)

# Critical supplier lot
sup_risk = df.groupby("critical_supplier_lot")[TARGET].value_counts(normalize=True).unstack().reindex(columns=risk_order, fill_value=0)
sup_risk.index = ["Standard Lot", "Critical Lot"]
sup_risk.plot(kind="bar", ax=axes[1], color=[risk_colors[r] for r in risk_order],
              edgecolor="white", rot=0)
axes[1].set_title("Risk by Supplier Lot")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Risk", fontsize=8)

# Setup checklist
chk_risk = df.groupby("setup_checklist_complete")[TARGET].value_counts(normalize=True).unstack().reindex(columns=risk_order, fill_value=0)
chk_risk.index = ["Incomplete", "Complete"]
chk_risk.plot(kind="bar", ax=axes[2], color=[risk_colors[r] for r in risk_order],
              edgecolor="white", rot=0)
axes[2].set_title("Risk by Setup Checklist")
axes[2].set_ylabel("Proportion")
axes[2].legend(title="Risk", fontsize=8)

plt.suptitle("Scrap Risk Rate by Structural Process Conditions",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter — operator experience vs press speed, colored by risk class
# Two most predictive continuous features — should show clear clustering
fig, ax = plt.subplots(figsize=(8, 6))
for cls in risk_order:
    mask = df[TARGET] == cls
    ax.scatter(df.loc[mask, "press_speed_spm"],
               df.loc[mask, "operator_experience_yrs"],
               alpha=0.35, s=16, color=risk_colors[cls], label=cls)
ax.set_xlabel("Press Speed (spm)")
ax.set_ylabel("Operator Experience (years)")
ax.set_title("Press Speed vs Operator Experience — Colored by Scrap Risk",
             fontweight="bold")
ax.legend(title="Scrap Risk")
plt.tight_layout()
plt.show()
print("Visible pattern: low experience + high speed concentrates High risk. Tree will use this.")


## Section 5 — Preprocessing

Decision Trees do not require feature scaling — they split on thresholds, not distances, so StandardScaler would have no effect on the model's decisions. The only preprocessing required is encoding the categorical variable `shift` (three levels: Day, Night, Early_Morning) via **OneHotEncoder with `drop='first'`**, and encoding the multiclass target with **LabelEncoder**.

The binary flags (`critical_supplier_lot`, `recent_model_change`, `setup_checklist_complete`) are already 0/1 integers — the tree handles them directly as numeric thresholds. Everything is bundled inside a **ColumnTransformer + Pipeline** for portability in the simulator.


In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# Encode target — LabelEncoder maps High→0, Low→1, Medium→2 alphabetically
le_y = LabelEncoder()
y_enc = le_y.fit_transform(y)
print("Label encoding:", dict(zip(le_y.classes_, le_y.transform(le_y.classes_))))

# ColumnTransformer: OHE for shift, passthrough for everything else
# Trees do not need scaling — only encoding
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", sparse_output=False), CAT_COLS),
    ("num", "passthrough", NUM_COLS)
])

# Stratified split — preserves class distribution in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.3, random_state=RANDOM_STATE, stratify=y_enc
)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"Classes      : {le_y.classes_}")


## Section 6 — Train Model

We train a Decision Tree with two explicit complexity controls: `max_depth=5` limits the tree to 5 levels from root to leaf, preventing overfitting while keeping the rules deep enough to capture the interaction structure of the problem. `min_samples_leaf=50` ensures each leaf represents at least 50 production runs — preventing the model from memorizing individual outliers that would not generalize to new runs.

These hyperparameters reflect a deliberate choice: in manufacturing analytics, a slightly less accurate but more generalizable model is always preferred to one that fits training data perfectly but fails on new production conditions.


In [ ]:
dt_clf = DecisionTreeClassifier(
    criterion="gini",         # Gini impurity — standard for classification
    max_depth=5,               # cap depth to prevent overfitting
    min_samples_leaf=50,       # each leaf must represent >= 50 production runs
    random_state=RANDOM_STATE
)

pipe_dt = Pipeline([
    ("preprocessor", preprocessor),
    ("model", dt_clf)
])

pipe_dt.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Algorithm      : DecisionTreeClassifier")
print(f"Criterion      : {dt_clf.criterion}")
print(f"Max depth      : {dt_clf.max_depth}")
print(f"Min samples/leaf: {dt_clf.min_samples_leaf}")
print(f"Actual depth   : {pipe_dt.named_steps['model'].get_depth()}")
print(f"Number of leaves: {pipe_dt.named_steps['model'].get_n_leaves()}")


## Section 7 — Evaluate

This is a three-class problem, so we need to look beyond overall accuracy. **Macro F1** averages F1 across all three classes with equal weight — it penalizes the model for performing poorly on any class, regardless of frequency. This is the appropriate metric when all three risk levels are operationally important: missing a High-risk run is critical, but so is systematically misclassifying Low-risk runs (which would trigger unnecessary interventions).

The per-class classification report reveals which risk levels the model handles well and where it struggles. The confusion matrix shows the exact error patterns — whether High gets confused with Medium, or Low gets confused with Medium, and in which direction.


In [ ]:
y_pred = pipe_dt.predict(X_test)
y_pred_train = pipe_dt.predict(X_train)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test, y_pred)
f1_macro  = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy  — Train : {acc_train:.4f}")
print(f"Accuracy  — Test  : {acc_test:.4f}")
print(f"F1 Macro  — Test  : {f1_macro:.4f}")
print(f"F1 Weighted — Test: {f1_weighted:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=le_y.classes_))


In [ ]:
# Confusion matrix — multiclass
cm = confusion_matrix(y_test, y_pred)
labels = le_y.classes_

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.set_title("Confusion Matrix — Scrap Risk (3 Classes)", fontweight="bold")
plt.tight_layout()
plt.show()
print("Rows = actual class. Columns = predicted class. Diagonal = correct predictions.")


In [ ]:
# Predicted probability distribution per class
y_prob_all = pipe_dt.predict_proba(X_test)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, cls in enumerate(le_y.classes_):
    color = risk_colors[cls]
    axes[i].hist(y_prob_all[y_test == i, i], bins=20, alpha=0.8,
                 color=color, edgecolor="white", label=f"True {cls}")
    axes[i].hist(y_prob_all[y_test != i, i], bins=20, alpha=0.5,
                 color="gray", edgecolor="white", label=f"Other classes")
    axes[i].set_title(f"P({cls}) Distribution", fontweight="bold")
    axes[i].set_xlabel(f"Predicted P({cls})")
    axes[i].set_ylabel("Count")
    axes[i].legend(fontsize=8)

plt.suptitle("Predicted Class Probabilities by True Class",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## Section 8 — Interpretability: Tree Visualization, Text Rules, and Feature Importance

Decision Tree interpretability comes in three complementary forms that serve different audiences:

1. **Tree visualization** — the graphic that operations managers and process engineers can read. Nodes show the split condition; leaves show the predicted class and confidence. Even a partial rendering of the first 3 levels reveals the dominant decision logic.

2. **Text rules** — the `export_text` output converts the tree into explicit if-then-else rules. These can be transcribed into process control documents, SOPs, or operator training materials.

3. **Gini feature importance** — measures how much each feature reduces impurity across all splits where it is used. This is the tree-native importance score: features that appear in higher nodes, with larger sample counts, accumulate more importance.


In [ ]:
# Recover full feature names after preprocessing
ohe = pipe_dt.named_steps["preprocessor"].named_transformers_["cat"]
cat_feature_names = list(ohe.get_feature_names_out(CAT_COLS))
all_feature_names = cat_feature_names + NUM_COLS

print("Encoded feature names (in model order):")
for i, name in enumerate(all_feature_names):
    print(f"  [{i}] {name}")


In [ ]:
# Tree visualization — first 3 levels only for readability
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    pipe_dt.named_steps["model"],
    feature_names=all_feature_names,
    class_names=le_y.classes_,
    filled=True,
    max_depth=3,
    fontsize=8,
    ax=ax
)
ax.set_title(
    "Decision Tree — First 3 Levels (of 5 total)\n"
    "Top nodes = most important splits. Leaf colors: Blue=High, Orange=Low, Green=Medium",
    fontweight="bold", fontsize=11
)
plt.tight_layout()
plt.show()


In [ ]:
# Text rules — first 80 lines of the full tree rules
# These can be transcribed directly into process control documents
tree_rules = export_text(
    pipe_dt.named_steps["model"],
    feature_names=all_feature_names,
    decimals=2
)
lines = tree_rules.split("\n")
print("Decision Tree — Operational Rules (first 80 lines):")
print("=" * 60)
print("\n".join(lines[:80]))
print("...")
print(f"[Total rule lines: {len(lines)}]")


In [ ]:
# Gini feature importance — sorted bar chart
importances = pipe_dt.named_steps["model"].feature_importances_
imp_df = pd.DataFrame({
    "Feature":    all_feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("Feature Importance (Gini-based — reduction in impurity):")
print(imp_df.round(4).to_string(index=False))


In [ ]:
# Gini importance bar chart
imp_sorted = imp_df.sort_values("Importance", ascending=True)
colors_imp = ["#E8574C" if v > 0.1 else "#4C9BE8" for v in imp_sorted["Importance"]]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(imp_sorted["Feature"].str.replace("_", " ").str.title(),
               imp_sorted["Importance"],
               color=colors_imp, edgecolor="white", height=0.65)
for bar, val in zip(bars, imp_sorted["Importance"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", ha="left", fontsize=9)
ax.set_xlabel("Gini Importance (reduction in impurity)")
ax.set_title("Feature Importance — Decision Tree (Gini)\nRed = dominant drivers",
             fontweight="bold")
ax.axvline(0.1, color="gray", linestyle="--", linewidth=1, label="10% threshold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 2D decision boundary map — press speed vs operator experience
# These are the top two continuous drivers — tree trained on only these two features
features_2d = ["press_speed_spm", "operator_experience_yrs"]

X_2d = df[features_2d]
X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_2d, y_enc, test_size=0.3, random_state=RANDOM_STATE, stratify=y_enc
)

dt_2d = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=RANDOM_STATE)
dt_2d.fit(X_2d_train, y_2d_train)

x_min, x_max = X_2d[features_2d[0]].min() - 2, X_2d[features_2d[0]].max() + 2
y_min, y_max = X_2d[features_2d[1]].min() - 0.5, X_2d[features_2d[1]].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = dt_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
# Map encoded class to color: 0=High(red), 1=Low(blue), 2=Medium(orange)
class_to_color = {0: "#E8574C", 1: "#4C9BE8", 2: "#F0A500"}
Z_rgb = np.zeros((*Z.shape, 3))
for cls, rgb in [(0, [0.91, 0.34, 0.30]), (1, [0.30, 0.61, 0.91]), (2, [0.94, 0.65, 0.0])]:
    mask_z = Z == cls
    Z_rgb[mask_z] = rgb

ax.imshow(Z_rgb, extent=[x_min, x_max, y_min, y_max],
          origin="lower", aspect="auto", alpha=0.35)

for cls in range(3):
    mask = y_enc == cls
    ax.scatter(X_2d.loc[mask, features_2d[0]], X_2d.loc[mask, features_2d[1]],
               color=list(class_to_color.values())[cls], s=12, alpha=0.45,
               label=le_y.classes_[cls])

ax.set_xlabel("Press Speed (spm)")
ax.set_ylabel("Operator Experience (years)")
ax.set_title("Decision Tree Boundary — Press Speed vs Operator Experience\n"
             "(other features held out — 2D simplification)", fontweight="bold")
ax.legend(title="Scrap Risk")
plt.tight_layout()
plt.show()
print(f"2D model accuracy: {accuracy_score(y_2d_test, dt_2d.predict(X_2d_test)):.4f}")


### Summary of Practical Signals

The Gini importance scores and tree structure tell a coherent process story — one that any stamping engineer would recognize:

- **`operator_experience_yrs` is the top driver (30.3%)** — operator maturity is the strongest single determinant of scrap risk in this dataset. Experienced operators compensate for difficult conditions; inexperienced ones amplify every other risk factor. This is not just a statistical signal — it is the foundational argument for structured skills development programs on high-speed lines.
- **`setup_checklist_complete` (21.7%)** — the checklist is the second most important variable. An incomplete pre-run checklist is the single most controllable intervention point: it costs nothing to require, and the model confirms its systematic impact on scrap risk.
- **`critical_supplier_lot` (20.0%)** — supplier material quality ranks third. This is largely outside the operator's control, which is exactly why supplier qualification and incoming inspection protocols exist. The model quantifies what quality engineers already know intuitively.
- **`press_speed_spm` (19.5%)** — press speed is the primary process parameter. High speed increases throughput but compounds risk, especially when combined with inexperienced operators or incomplete setup.
- **`recent_model_change` (7.8%)** — changeover events are a known scrap trigger in stamping. The model correctly captures their impact, though smaller than the four leading factors.
- **`raw_material_hardness_hrb`, `ambient_temp_c`, and shift variables** — these contribute minimally within the simulated data ranges, though in real production they would likely carry more weight under extreme conditions.

**Operational implication:** The top four variables — experience, checklist, supplier lot, and speed — account for 87% of the predictive signal. A process control improvement focused on just these four levers would capture the vast majority of preventable scrap risk.


## Section 9 — Statistical Validation

With **2,317 samples** and three classes, a single 70/30 split can still be noisy for minority classes. We validate stability with 5-fold stratified cross-validation — which preserves the class distribution in each fold — and report the confidence interval for test-set accuracy.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc  = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring="accuracy")
cv_f1   = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring="f1_macro")

print("5-Fold Stratified Cross-Validation (training set only):")
print(f"  Accuracy  : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  F1 Macro  : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")


In [ ]:
# 95% confidence interval for test-set accuracy
n = len(y_test)
p = accuracy_score(y_test, y_pred)
z = 1.96
margin = z * np.sqrt((p * (1 - p)) / n)

print(f"Test-set accuracy : {p:.4f}")
print(f"95% CI            : [{p - margin:.4f}, {p + margin:.4f}]")
print(f"Margin of error   : ±{margin:.4f}")
print()
print("Train/Test gap analysis:")
print(f"  Train accuracy: {acc_train:.4f}")
print(f"  Test  accuracy: {acc_test:.4f}")
print(f"  Gap           : {acc_train - acc_test:.4f}")
print()
print("Small gap = good generalization. Large gap = overfitting.")


## Section 10 — Production Scenario Simulator

A simulator converts the model into a pre-run risk assessment tool. Before starting a production order, a process engineer or shift supervisor can enter the current conditions — press speed, material hardness, operator experience, supplier lot status, checklist completion — and receive a three-class risk probability. This enables data-driven go/no-go decisions before the run begins, not after scrap is already in the bin.


In [ ]:
def simulate_production_run(
    press_speed_spm: float = 35.0,
    raw_material_hardness_hrb: float = 80.0,
    operator_experience_yrs: float = 5.0,
    ambient_temp_c: float = 24.0,
    critical_supplier_lot: int = 0,
    shift: str = "Day",
    recent_model_change: int = 0,
    setup_checklist_complete: int = 1
) -> tuple:
    """
    Estimate scrap risk class for a production run before it begins.

    Parameters
    ----------
    press_speed_spm           : Press speed in strokes per minute (range: 20–60)
    raw_material_hardness_hrb : Raw material hardness in HRB (range: 60–100, mean ~80)
    operator_experience_yrs   : Operator years of experience (0–20)
    ambient_temp_c            : Shop floor ambient temperature °C (typical: 15–35)
    critical_supplier_lot     : 0 = standard lot, 1 = lot from supplier with defect history
    shift                     : 'Day', 'Night', or 'Early_Morning'
    recent_model_change        : 0 = same model, 1 = model/die change in this run
    setup_checklist_complete  : 0 = incomplete pre-run checklist, 1 = fully completed
    """
    df_new = pd.DataFrame([{
        "press_speed_spm":           press_speed_spm,
        "raw_material_hardness_hrb": raw_material_hardness_hrb,
        "operator_experience_yrs":   operator_experience_yrs,
        "ambient_temp_c":            ambient_temp_c,
        "critical_supplier_lot":     critical_supplier_lot,
        "shift":                     shift,
        "recent_model_change":        recent_model_change,
        "setup_checklist_complete":   setup_checklist_complete
    }])

    pred_enc = pipe_dt.predict(df_new)[0]
    proba    = pipe_dt.predict_proba(df_new)[0]
    clase    = le_y.inverse_transform([pred_enc])[0]

    print("=" * 58)
    print("  PRODUCTION RUN SCRAP RISK SIMULATOR")
    print("=" * 58)
    print(f"  Press speed           : {press_speed_spm:.0f} spm")
    print(f"  Material hardness     : {raw_material_hardness_hrb:.0f} HRB")
    print(f"  Operator experience   : {operator_experience_yrs:.1f} yrs")
    print(f"  Ambient temperature   : {ambient_temp_c:.1f} °C")
    print(f"  Critical supplier lot : {'YES' if critical_supplier_lot else 'No'}")
    print(f"  Shift                 : {shift.upper()}")
    print(f"  Recent model change   : {'YES' if recent_model_change else 'No'}")
    print(f"  Setup checklist       : {'Complete' if setup_checklist_complete else 'INCOMPLETE'}")
    print("-" * 58)
    print(f"  Predicted scrap risk  : >>> {clase.upper()} <<<")
    print("  Class probabilities:")
    for cls, p in zip(le_y.classes_, proba):
        bar = "█" * int(p * 30)
        print(f"    {cls:<8} {p:.3f}  {bar}")
    print("=" * 58)
    return clase, dict(zip(le_y.classes_, proba))


### Scenario A — Low-risk: experienced operator, complete checklist, standard conditions


In [ ]:
simulate_production_run(
    press_speed_spm=30,
    raw_material_hardness_hrb=80,
    operator_experience_yrs=10,
    ambient_temp_c=24,
    critical_supplier_lot=0,
    shift="Day",
    recent_model_change=0,
    setup_checklist_complete=1
)


### Scenario B — High-risk: new operator, critical supplier, incomplete checklist, high speed


In [ ]:
simulate_production_run(
    press_speed_spm=55,
    raw_material_hardness_hrb=92,
    operator_experience_yrs=0.5,
    ambient_temp_c=32,
    critical_supplier_lot=1,
    shift="Night",
    recent_model_change=1,
    setup_checklist_complete=0
)


### Scenario C — What-if: same high-risk profile, but checklist completed and speed reduced


In [ ]:
# Isolates the effect of completing the checklist and reducing speed
# on an otherwise high-risk run
simulate_production_run(
    press_speed_spm=38,         # speed reduced to moderate level
    raw_material_hardness_hrb=92,
    operator_experience_yrs=0.5,
    ambient_temp_c=32,
    critical_supplier_lot=1,
    shift="Night",
    recent_model_change=1,
    setup_checklist_complete=1  # checklist completed
)


## Final Reflection

This project is not about predicting scrap with certainty.  
It is about **making the risk of a bad run visible before the press starts**.

Scrap in stamping is not random. It is the compound result of a specific set of conditions converging — a new operator on a night shift with a difficult material lot and an incomplete pre-run setup. Each of these factors alone is manageable. The combination is not. Decision Trees excel at capturing this compound logic: they identify which combination of conditions crosses a threshold from acceptable to risky, and they express it in language that an industrial engineer can read, challenge, and act on.

What this model offers is not a verdict, but a **conversation**:
- *What press speed can we safely run with this operator and this material lot?*
- *What is the risk difference between completing and skipping the setup checklist?*
- *Which combinations of process conditions create a Low-risk corridor we can defend?*

These questions matter more than the accuracy score.

---

*LozanoLsa  |  Operational Excellence · MBB · Machine Learning  |  GitHub: LozanoLsa*
